# Day 044 · akshare 拉 A 股
**akshare** · 阶段 P2 · Python 量化工具栈

> 上一节的 yfinance 拉海外很方便,但一到 A 股就不灵了——美国网站对沪深两市的行情和财报支持很差。这一节请出做国内数据的瑞士军刀 akshare:免费、不用 token,一套接口就能拉 A 股行情、沪深 300 成分股权重、个股财务报表,还能拉港股、期货、基金、宏观甚至另类数据。我们用它把中国平安和沪深 300 的数据一次备齐,顺便讲清楚国内免费数据最大的脾气——偶尔断连,所以代码必须学会重试和限速。

---

**课件生成日期:** 2026-06-14  ·  **建议学习时长:** 18 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [ ]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["akshare", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


## 学习目标

- 知道 akshare 是免费拉国内数据(A 股/港股/期货/基金/宏观)的瑞士军刀,无需注册和 token
- 会用 ak.stock_zh_a_hist 一行拉 A 股个股的前复权日线行情
- 会用 ak.index_stock_cons_weight_csindex 拉沪深 300 成分股和它们的权重
- 会用 ak.stock_financial_abstract 拉个股财务摘要(营收、净利润等)
- 会写 _retry 重试 + time.sleep 限速,应对国内免费数据偶尔断连的脾气

## 历史背景:老陈做 A 股选股,被 Wind 的报价单劝退了

老陈想认真做一次 A 股基本面选股:先拿到沪深 300 的成分名单和权重,再翻几家龙头的财报。他先去问专业数据终端 Wind,销售报了个一年好几万的价格,老陈当场倒吸一口凉气——他一个散户,做点研究而已,哪掏得起这个钱。他又试着去各财经网站一个个手抄,抄到一半发现财报字段又多又乱,还容易抄错。正发愁时,群里一位写代码的朋友甩给他三行 akshare:一行拉沪深 300 成分权重、一行拉中国平安5 年行情、一行拉财务摘要,全部免费、几秒到手。老陈这才发现,散户和机构在数据上的差距,早就被这种免费开源工具抹平了一大半。唯一要注意的是,免费数据连的是公开网站,偶尔会断,所以代码得学会“失败了等几秒再试一次”——这点小脾气,包一层重试就治好了。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. akshare:国内数据的瑞士军刀

akshare 是一个开源的中文金融数据包,免费、不用注册、不用申请 token。它把各大公开数据源封装成一个个函数:A 股行情、指数成分、个股财报、港股美股、期货期权、基金、宏观经济、甚至新闻舆情这类另类数据,几乎想要啥都有一个对应函数。对做 A 股研究的散户来说,它几乎抹平了和机构在数据上的差距。


### 2. stock_zh_a_hist:一行拉 A 股个股行情

ak.stock_zh_a_hist(symbol='601318', period='daily', adjust='qfq') 一行就拿到中国平安的日线行情。symbol 是6 位代码,period 可选 daily/weekly/monthly,adjust 控制复权:'qfq' 前复权、'hfq' 后复权、''(空)不复权。和上一节的道理一样,算收益要用复权价。返回的是中文列名(日期/开盘/收盘…),用前先改成习惯的英文名再处理。


### 3. index_stock_cons_weight_csindex:指数成分和权重

想知道沪深 300 里都有哪些股票、各占多大分量,用 ak.index_stock_cons_weight_csindex(symbol='000300'),返回 300 只成分券和它们的权重。权重越大的股票,对指数涨跌的影响越大——这就是为什么少数几只大白马一动,整个指数就跟着抖。这份名单也是很多选股、做指数增强策略的起点。


### 4. stock_financial_abstract:个股财务摘要

ak.stock_financial_abstract(symbol='601318') 拉一家公司的财务摘要,营业收入、净利润、每股收益、净资产收益率等关键指标按报告期排开。这是基本面分析的原料。要注意它是一张宽表(行是指标、列是一个个报告期),不同 akshare 版本列名会有细微差别,解析时包一层 try/except、先打印出来看清结构再取值,最稳妥。


### 5. _retry 重试 + 限速:免费数据的必修课

akshare 背后连的是公开网站,不像收费终端那么稳,偶尔会超时、断连。正确做法是把每个拉取调用包进一个重试函数:失败了等几秒再试,试几次都不行才报错,别一次抖动就让整节代码挂掉。批量拉很多标的时,还要 time.sleep 限速,别把人家网站请求爆了。这两招是用任何免费数据源的基本功。


## 实操:akshare — 一行拉 A 股行情 / 沪深 300 成分与权重 / 个股财务摘要 / _retry 重试 + 限速

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [ ]:
# day_044_akshare.py — akshare:国内数据瑞士军刀 / 一段代码拉 A 股行情 / 沪深 300 成分与权重 / 个股财务 / _retry 重试 + 限速
# 用 akshare 免费拉沪深 300 成分股权重、拉中国平安五年行情和财务摘要,做一次零成本的 A 股数据备齐
# 数据:akshare(免费拉 A 股/港股/美股/期货/基金/宏观/另类,无需 token)
# 铁律 62:拉数据的课把数据存 CSV 到仓库根 data/,load 优先 + fetch 兜底 + 自动 to_csv
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import akshare as ak
from pathlib import Path
def _data_path(_name):
    # 铁律62:data/ 放在 notebook 文件夹里(run_lab 存取 out/notebook/data/)
    from pathlib import Path as _P
    _here = _P.cwd()
    for _b in [_here/'data', _here/'..'/'data', _here/'out'/'notebook'/'data', _here/'..'/'..'/'data', _here/'..'/'..'/'..'/'data']:
        if (_b/_name).exists():
            return str(_b/_name)
    if (_here/'out'/'notebook').exists():
        _t = _here/'out'/'notebook'/'data'
    elif _here.name == 'cn':
        _t = _here/'..'/'data'
    else:
        _t = _here/'data'
    _t.mkdir(parents=True, exist_ok=True)
    return str(_t/_name)

pd.set_option('display.width', 160)
plt.rcParams['axes.unicode_minus'] = False



def _retry(fn, *args, tries=3, sleep=2, **kwargs):
    """akshare 连公开网站偶尔断连,包一层重试:失败等几秒再来,别一次失败就整节挂掉。"""
    last = None
    for i in range(tries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last = e
            print(f'  第 {i+1}/{tries} 次失败:{type(e).__name__},{sleep}s 后重试...')
            time.sleep(sleep)
    raise RuntimeError(f'{fn.__name__} 重试 {tries} 次仍失败:{last}')


def load_or_fetch(csv_path, fetch_fn, **read_kwargs):
    """铁律 62:CSV 在就直接读,不在才联网拉 + 落盘,下次零成本复跑。"""
    csv_path = Path(csv_path)
    if csv_path.exists():
        print(f'  载入本地 CSV:{csv_path}')
        return pd.read_csv(csv_path, **read_kwargs)
    print(f'  本地无 {csv_path.name},联网拉取并保存...')
    df = fetch_fn()
    df.to_csv(csv_path)
    print(f'  已保存:{csv_path}')
    return df


# ====================================================================
print('==== 1. 沪深 300 成分股 + 权重(谁在指数里占多大分量)====')
w = load_or_fetch(Path(_data_path('D044_akshare_weight.csv')),
                  lambda: _retry(ak.index_stock_cons_weight_csindex, symbol='000300'),
                  index_col=0)
print(f'沪深 300 成分券 {len(w)} 只,列:{list(w.columns)}')
wcol = '权重' if '权重' in w.columns else w.columns[-1]
ncol = '成分券名称' if '成分券名称' in w.columns else [c for c in w.columns if '名称' in c][0]
top = w.sort_values(wcol, ascending=False).head(15)
print('权重最大的 15 只:')
print(top[[ncol, wcol]].to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top[ncol][::-1], top[wcol][::-1].astype(float), color='#5e81ac')
ax.set_title('沪深 300 权重前 15 名(指数里谁说了算)'); ax.set_xlabel('权重 %')
plt.tight_layout(); plt.savefig('chart_01.png', dpi=120); plt.close()
print('图1(沪深 300 权重前 15)已保存')

# ====================================================================
print('\n==== 2. 个股行情:中国平安 五年日线(前复权)====')


def _fetch_price():
    raw = _retry(ak.stock_zh_a_hist, symbol='601318', period='daily',
                 start_date='20200101', end_date='20241231', adjust='qfq')
    out = raw.rename(columns={'日期': 'date', '开盘': 'Open', '最高': 'High',
                              '最低': 'Low', '收盘': 'Close', '成交量': 'Volume'})
    out['date'] = pd.to_datetime(out['date'])
    return out.set_index('date').sort_index()


df = load_or_fetch(Path(_data_path('D044_akshare_price.csv')), _fetch_price,
                   index_col=0, parse_dates=True)
for c in ['Open', 'High', 'Low', 'Close', 'Volume']:
    df[c] = df[c].astype(float)
print(f'中国平安 日线 {len(df)} 条  {df.index[0].date()} → {df.index[-1].date()}')
print(df[['Open', 'High', 'Low', 'Close', 'Volume']].tail(3).round(2))

fig, (axp, axv) = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})
axp.plot(df.index, df['Close'], color='#4c566a', lw=1.1, label='收盘')
axp.plot(df.index, df['Close'].rolling(60).mean(), color='#d08770', lw=1.1, label='60 日均线')
axp.set_title('中国平安 收盘 + 60 日均线 / 成交量'); axp.set_ylabel('价格(元)'); axp.legend()
axv.bar(df.index, df['Volume'] / 1e6, color='#b48ead', width=1.0)
axv.set_ylabel('成交量(百万手)')
plt.tight_layout(); plt.savefig('chart_02.png', dpi=120); plt.close()
print('图2(中国平安 价格 + 成交量)已保存')

# ====================================================================
print('\n==== 3. 个股财务:中国平安 财务摘要(营收/净利润等)====')
fa = load_or_fetch(Path(_data_path('D044_akshare_fin.csv')),
                   lambda: _retry(ak.stock_financial_abstract, symbol='601318'),
                   index_col=0)
print(f'财务摘要表 {fa.shape[0]} 行 × {fa.shape[1]} 列,列:{list(fa.columns)[:6]} ...')
print(fa.head(6).to_string(index=False))

# 财务表是宽表(行=指标,列=报告期),版本不同列名略有差异,稳妥地包 try/except
try:
    indcol = [c for c in fa.columns if '指标' in c][0]
    date_cols = [c for c in fa.columns if str(c).isdigit() and len(str(c)) == 8][:10]  # 报告期列形如 20231231
    date_cols = sorted(date_cols)
    rev_row = fa[fa[indcol].astype(str).str.contains('营业总收入|营业收入', na=False)].iloc[0]
    profit_row = fa[fa[indcol].astype(str).str.contains('归母净利润|净利润', na=False)].iloc[0]
    rev = pd.to_numeric(rev_row[date_cols], errors='coerce') / 1e8       # 转亿元
    profit = pd.to_numeric(profit_row[date_cols], errors='coerce') / 1e8
    xlab = [c[:4] + '/' + c[4:6] for c in date_cols]
    fig, ax = plt.subplots(figsize=(12, 5.5))
    x = np.arange(len(date_cols))
    ax.bar(x - 0.2, rev.values, width=0.4, color='#5e81ac', label='营业收入(亿元)')
    ax.bar(x + 0.2, profit.values, width=0.4, color='#bf616a', label='净利润(亿元)')
    ax.set_xticks(x); ax.set_xticklabels(xlab, rotation=30, ha='right')
    ax.set_title('中国平安 营收与净利润(近若干报告期)'); ax.legend()
    plt.tight_layout(); plt.savefig('chart_03.png', dpi=120); plt.close()
    print('图3(营收/净利润)已保存')
except Exception as e:
    print(f'财务字段解析需按实际列名微调({type(e).__name__}: {e}),已先打印原表供核对')

print('\n[小结] akshare 一套接口把指数成分、个股行情、财务摘要全拉齐,零成本零 token')
print('[done] A 股数据备齐,output.txt 见上方全部打印')

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| A 股 | 601318 | 中国平安5 年前复权日线行情,收盘 + 60 日均线 + 成交量一张图看趋势 |
| 指数成分 | 000300 | 沪深 300 全部成分股权重,排出前 15 大权重股,看谁对指数说了算 |
| 基本面 | 601318 | 财务摘要拉营收、净利润历年趋势,做基本面选股的第一手原料 |
| 多市场拓展 | 港股/期货/基金 | 换个函数就能拉港股、期货、基金、宏观,一个 akshare 把国内数据备齐 |


## 常见坑

### ⚠ 01. 不重试,一断连整节挂

akshare 连公开网站偶尔超时断连,直接裸调一抖动就报错、整个 notebook 停在半路。必须包 _retry:失败等几秒重试几次。这是免费数据源的头号坑,也是最容易忽略的。

### ⚠ 02. 中文列名直接拿去算

akshare 返回的是中文列名(日期/开盘/收盘/成交量),直接拿去和习惯英文列名的代码拼会对不上。拉完先 rename 成 Open/High/Low/Close/Volume,再 to_datetime + set_index,后续才顺。

### ⚠ 03. 复权参数选错算错收益

stock_zh_a_hist 的 adjust 默认是空字符串即不复权,直接拿去算长期收益会被分红除权骗。做收益、回测要显式写 adjust='qfq'(前复权)或 'hfq'(后复权),别用默认的不复权价。

### ⚠ 04. 财务宽表硬取列名报错

财务摘要是宽表,不同版本的 akshare 列名、指标措辞会变(“营业总收入”还是“营业收入”)。别写死列名硬取,先打印 columns 和前几行看清结构,用包含匹配 + try/except 容错地取,别让一个字段名变动就崩掉。

### ⚠ 05. 批量猛拉不限速

和 yfinance 一样,批量拉几百只成分股时连续猛请求会被网站限流。循环里 time.sleep 歇一下,失败重试,温柔地拿数据,才能稳定跑完。

## 实战 SOP · akshare 拉 A 股实战 6 条 SOP

1. 每个拉取都包 _retry — 失败等几秒重试,别一断连就整节挂。
2. 中文列名先 rename — 改成 Open/High/Low/Close/Volume 再处理。
3. 复权显式写 adjust='qfq' — 算收益别用默认的不复权价。
4. 财务宽表先打印再取 — 包含匹配 + try/except,别写死列名。
5. 批量拉必 time.sleep 限速 — 温柔请求,别被网站限流。
6. 国内联网也要 CSV 回退 — 行情包 try/except,失败读本地预下载数据(铁律 55)。

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. akshare 是免费、免 token 的国内数据瑞士军刀:A 股/港股/期货/基金/宏观/另类几乎全有。
3. ak.stock_zh_a_hist 一行拉个股行情,记得 adjust='qfq' 用复权价。
4. ak.index_stock_cons_weight_csindex 拉沪深 300 成分和权重,看谁对指数说了算。
5. ak.stock_financial_abstract 拉财务摘要,是基本面分析的原料,宽表要容错解析。
6. 免费数据偶尔断连,必须 _retry 重试 + time.sleep 限速,这是基本功。
7. 中国版同样包 try/except + 本地 CSV 回退,联网失败也能跑完(铁律 55)。

## 自测题

**Q1.** akshare 和上一节的 yfinance 分工上有什么不同?(提示:yfinance 强在拉海外(美股、港股等),A 股的行情和财报支持差;akshare 专精国内,免费拉 A 股行情、指数成分、个股财报、宏观和另类数据,做 A 股研究用它。)

**Q2.** 用 akshare 拉中国平安的复权日线怎么写,要注意什么?(提示:ak.stock_zh_a_hist(symbol='601318', period='daily', adjust='qfq');adjust 默认是空字符串不复权,算收益必须显式写 'qfq' 或 'hfq';返回中文列名,要先 rename 再处理。)

**Q3.** 为什么用 akshare 几乎必须写重试?(提示:它连的是公开网站,不像收费终端稳定,偶尔超时断连;裸调一抖动就报错、整节挂掉。包一个 _retry 函数失败等几秒再试几次,加上 time.sleep 限速,才能稳定拿数据。)

**Q4.** 想知道沪深 300 里哪些股票权重最大,用哪个函数?权重大意味着什么?(提示:ak.index_stock_cons_weight_csindex(symbol='000300') 拉成分和权重;权重越大对指数涨跌影响越大,所以少数几只大白马一动整个指数就跟着抖。)

把答案写下来,3 天后再回看。

## 下一节预告

**Day 045 · SQLite 存行情** (SQLite)

学完 yfinance 和 akshare,你已经能免费把海内外行情和财报拉齐了。但每次重新联网拉数据又慢又怕断,下一节我们学怎么把拉回来的行情存进本地数据库 SQLite:建表、写入、加索引,以后查数据一句 SQL 秒出,再不用每次都去联网。

## 推荐阅读

- akshare 官方文档(akfamily, 2024)— 股票/指数/财务/宏观各接口的函数名与参数权威参考
- akshare 官方文档《股票数据 - 历史行情》(2024)— stock_zh_a_hist 复权参数与字段说明
- 中证指数公司《沪深 300 指数编制方案》(2024)— 成分股筛选与权重计算规则,理解指数代表什么
- Wes McKinney《Python for Data Analysis》(2022, O'Reilly)— 宽表、长表转换与缺失处理,解析财务表必备
- Aswath Damodaran《Investment Valuation》(2012, Wiley)— 营收、利润、ROE 等财务指标在估值里的意义